In [ ]:
import torch
import matplotlib.pyplot as plt

from fim.physics.diffusion import DiffusionBenchmark, DiffusionConfig
from fim.physics.wave import WaveBenchmark, WaveConfig
from fim.physics.chaotic import Lorenz96Benchmark, Lorenz96Config

In [ ]:
cfg = DiffusionConfig(diffusivity=0.1, dt=0.1, steps=50)
model = DiffusionBenchmark(cfg)

u0 = model.sample_initial_state(1, 32, 32)

traj = model.rollout(u0)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, t in enumerate([0, 10, 20, 30, 50]):
    axes[i].imshow(traj[0, t, 0].numpy())
    axes[i].set_title(f"t={t}")
    axes[i].axis("off")

plt.show()

In [ ]:
cfg = WaveConfig(speed=1.0, dt=0.05, steps=50)
model = WaveBenchmark(cfg)

u0, v0 = model.sample_initial_state(1, 32, 32)

traj = model.rollout(u0, v0)

plt.plot([traj[0, t].abs().mean().item() for t in range(50)])
plt.title("Wave Energy")
plt.show()

In [ ]:
cfg = Lorenz96Config(dimension=32, steps=100)
model = Lorenz96Benchmark(cfg)

x0 = model.sample_initial_state(1)

traj = model.rollout(x0)

plt.plot(traj[0].cpu().numpy())
plt.title("Lorenz-96 Trajectories")
plt.show()

In [ ]:
from fim.models.fim_model import FIMModel

model = FIMModel(
    in_channels=1,
    out_channels=1,
    latent_channels=16,
    trace_dim=8,
    hidden_channels=32,
).to("cpu")

x = torch.randn(1, 1, 32, 32)

states = []

with torch.no_grad():
    for _ in range(20):
        y, out = model.step(x, update_memory=True)
        states.append(out.state[0, 0].cpu())
        x = y

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, t in enumerate([0, 5, 10, 15, 19]):
    axes[i].imshow(states[t])
    axes[i].set_title(f"step {t}")
    axes[i].axis("off")

plt.show()